In [12]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [13]:
%reload_ext autoreload
%autoreload 2

In [14]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [15]:
from src.models.gcn import GCNModel, train_gcn, evaluate_gcn, create_loader
import numpy as np
import torch
import torch.nn as nn

In [16]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

## Sanity check — quick single run
Run this first to confirm the model loads, trains, and evaluates without errors before launching the full grid search.

In [17]:
from src.losses.focal_loss import FocalLoss
from torch_geometric import EdgeIndex

node_features = np.load("../data/processed/graph/node_features.npy")
edge_index = np.load("../data/processed/graph/edge_index.npy")

# Validate edge index — same as gat.ipynb
tg_edge_index = EdgeIndex(edge_index)
edge_index = tg_edge_index.validate().detach().clone().to(dtype=torch.long)

in_channels = node_features.shape[1]  # 166 for Elliptic
out_channels = 2                       # binary: licit vs illicit
hidden_channels = 64

X = np.load("../data/processed/baseline/X.npy")
y = np.load("../data/processed/baseline/y.npy")

mask = y >= 0
print(f"Known labelled nodes: {mask.sum()} / {len(y)}")
print(f"Total nodes: {len(X)}")

data, cls_weights = create_loader(X, y, edge_index)
print(f"Class counts (licit, illicit): {cls_weights}")

# GCN has no heads parameter — simpler than GAT
model = GCNModel(in_channels, hidden_channels, out_channels)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), weight_decay=0.0005)
criterion = FocalLoss(weight=cls_weights)

model = train_gcn(model, data, optimizer, criterion, epochs=10)
total_eval_loss, class_report_dict, class_report = evaluate_gcn(model, data, criterion)
print(class_report)

Known labelled nodes: 46564 / 203769
Total nodes: 203769
Class counts (licit, illicit): [33179  4072]
[0 1]
Epoch 1 / 10, Loss: 0.8874
Epoch 2 / 10, Loss: 0.4211
Epoch 3 / 10, Loss: 0.3697
Epoch 4 / 10, Loss: 0.3470
Epoch 5 / 10, Loss: 0.3343
Epoch 6 / 10, Loss: 0.3139
Epoch 7 / 10, Loss: 0.2927
Epoch 8 / 10, Loss: 0.2853
Epoch 9 / 10, Loss: 0.2730
Epoch 10 / 10, Loss: 0.2623
Validation loss: 0.2559675872325897
Validation F1 (illicit): 0.2187
              precision    recall  f1-score   support

       licit       0.97      0.79      0.87      8840
     illicit       0.13      0.59      0.22       473

    accuracy                           0.78      9313
   macro avg       0.55      0.69      0.55      9313
weighted avg       0.93      0.78      0.84      9313



## Grid search — hyperparameter tuning
GCN has fewer hyperparameters than GAT (no attention heads), so the search space is smaller and faster to run.
Best config is tracked by illicit F1-score and saved to `grid_search_gcn.txt`.

In [ ]:
from src.losses.focal_loss import FocalLoss
from itertools import product
import gc

# Optimizer hyperparameters
learning_rates = [0.01, 0.001]
weight_decays = [0.01, 0.001, 0.0001]

# GCN hyperparameters (no heads — unlike GAT)
dropout_rates = [0.5, 0.6, 0.8]
hidden_channels_opts = [32, 64, 128]

# Focal loss hyperparameters
gamma = [0, 0.5, 1, 2, 5]
reweight_beta = [0.9, 0.999, 0.9999]

X = np.load("../data/processed/baseline/X.npy")
y = np.load("../data/processed/baseline/y.npy")
data, cls_weights = create_loader(X, y, edge_index)

best_f1 = 0.0
best_hyperparams = {}

num_configs = (
    len(learning_rates) *
    len(weight_decays) *
    len(dropout_rates) *
    len(hidden_channels_opts) *
    len(gamma) *
    len(reweight_beta)
)
print(f"Total configs to search: {num_configs}")

count = 1
hyperparams = product(learning_rates, weight_decays, dropout_rates, hidden_channels_opts, gamma, reweight_beta)

for learning_rate, weight_decay, dropout, hidden, g, beta in hyperparams:
    print(f"Config {count} of {num_configs}")
    count += 1

    model = GCNModel(in_channels, hidden, out_channels, dropout=dropout)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.999), weight_decay=weight_decay)
    criterion = FocalLoss(weight=cls_weights, gamma=g, beta=beta)

    model = train_gcn(model, data, optimizer, criterion, epochs=50)
    total_eval_loss, class_report_dict, class_report = evaluate_gcn(model, data, criterion)

    illicit_f1 = class_report_dict['illicit']['f1-score']
    if illicit_f1 > best_f1:
        print("New best found!")
        best_f1 = illicit_f1
        best_hyperparams = {
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "dropout": dropout,
            "hidden_channels": hidden,
            "gamma": g,
            "beta": beta
        }
        print(best_hyperparams)
        print(class_report)

        with open("grid_search_gcn.txt", "a") as f:
            f.write("New best found!\n")
            f.write(f"learning_rate = {learning_rate}\n")
            f.write(f"weight_decay = {weight_decay}\n")
            f.write(f"dropout = {dropout}\n")
            f.write(f"hidden_channels = {hidden}\n")
            f.write(f"gamma = {g}\n")
            f.write(f"beta = {beta}\n")
            f.write(f"Best F1 Score: {best_f1}\n")
            f.write(class_report)
            f.write("\n---------------------\n\n")

    del model, optimizer, criterion, total_eval_loss
    torch.cuda.empty_cache()
    gc.collect()

Total configs to search: 810
Config 1 of 810
[0 1]
Epoch 1 / 50, Loss: 1.8529
Epoch 2 / 50, Loss: 0.8809
Epoch 3 / 50, Loss: 0.5997
Epoch 4 / 50, Loss: 0.4291
Epoch 5 / 50, Loss: 0.3588
Epoch 6 / 50, Loss: 0.3284
Epoch 7 / 50, Loss: 0.3108
Epoch 8 / 50, Loss: 0.3029
Epoch 9 / 50, Loss: 0.2971
Epoch 10 / 50, Loss: 0.2952
Epoch 11 / 50, Loss: 0.2864
Epoch 12 / 50, Loss: 0.2789
Epoch 13 / 50, Loss: 0.2763
Epoch 14 / 50, Loss: 0.2722
Epoch 15 / 50, Loss: 0.2700
Epoch 16 / 50, Loss: 0.2666
Epoch 17 / 50, Loss: 0.2629
Epoch 18 / 50, Loss: 0.2616
Epoch 19 / 50, Loss: 0.2586
Epoch 20 / 50, Loss: 0.2568
Epoch 21 / 50, Loss: 0.2522
Epoch 22 / 50, Loss: 0.2483
Epoch 23 / 50, Loss: 0.2494
Epoch 24 / 50, Loss: 0.2481
Epoch 25 / 50, Loss: 0.2453
Epoch 26 / 50, Loss: 0.2426
Epoch 27 / 50, Loss: 0.2412
Epoch 28 / 50, Loss: 0.2424
Epoch 29 / 50, Loss: 0.2389
Epoch 30 / 50, Loss: 0.2375
Epoch 31 / 50, Loss: 0.2364
Epoch 32 / 50, Loss: 0.2366
Epoch 33 / 50, Loss: 0.2352
Epoch 34 / 50, Loss: 0.2349
Epoch 

In [11]:
# Inspect best config found
best_hyperparams

NameError: name 'best_hyperparams' is not defined

## Final run — best config
Train with the best hyperparameters found above for more epochs to get the final reported F1.

In [7]:
from src.models.gcn import GCNModel, train_gcn, evaluate_gcn, create_loader

# Plug in your best hyperparams from the grid search above
best_hidden = best_hyperparams["hidden_channels"]
best_dropout = best_hyperparams["dropout"]
best_lr = best_hyperparams["learning_rate"]
best_wd = best_hyperparams["weight_decay"]
best_gamma = best_hyperparams["gamma"]
best_beta = best_hyperparams["beta"]

data, cls_weights = create_loader(X, y, edge_index)
total_known = cls_weights.sum()
cls_weight_ratio = cls_weights / total_known
print(f"Class weights: {cls_weight_ratio}")

model = GCNModel(in_channels, best_hidden, out_channels, dropout=best_dropout)
optimizer = torch.optim.Adam(model.parameters(), lr=best_lr, betas=(0.9, 0.999), weight_decay=best_wd)
criterion = FocalLoss(weight=cls_weight_ratio, gamma=best_gamma, beta=best_beta)

train_gcn(model, data, optimizer, criterion, epochs=200)
total_eval_loss, class_report_dict, class_report = evaluate_gcn(model, data, criterion)
print(class_report)
print(f"\nFinal illicit F1: {class_report_dict['illicit']['f1-score']:.4f}")
print(f"Weber et al. baseline: 0.6280")

NameError: name 'best_hyperparams' is not defined

In [20]:
from src.models.gcn import GCNModel, train_gcn, evaluate_gcn, create_loader
from src.losses.focal_loss import FocalLoss

data, cls_weights = create_loader(X, y, edge_index)

# More aggressive class weight
illicit_weight = torch.tensor([29413, 3181], dtype=torch.float32)

model = GCNModel(in_channels, 128, out_channels, dropout=0.6)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999), weight_decay=0.001)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10
)

criterion = FocalLoss(weight=illicit_weight, gamma=2, beta=0.9999)

# Modified training loop with scheduler
model = model.to('cuda')
source = data.x.to('cuda')
target = data.y.to('cuda')
data_edge_index = data.edge_index.to('cuda')
train_mask = data.train_mask.to('cuda')

for epoch in range(150):
    model.train()
    optimizer.zero_grad()
    out = model(source, data_edge_index)
    loss = criterion(out[train_mask], target[train_mask])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step(loss)  # adjust lr when loss plateaus
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/150, Loss: {loss.item():.4f}, LR: {optimizer.param_groups[0]['lr']:.6f}")

del source, target, data_edge_index, out, loss, train_mask
torch.cuda.empty_cache()

# Threshold search
model.eval()
source = data.x.to('cuda')
target = data.y.to('cuda')
test_mask = data.test_mask.to('cuda')

with torch.no_grad():
    out = model(source, data.edge_index.to('cuda'))
    probs = torch.softmax(out[test_mask], dim=1)
    illicit_prob = probs[:, 1].cpu().numpy()
    true_labels = target[test_mask].cpu().numpy()

from sklearn.metrics import classification_report
best_f1, best_thresh = 0, 0.5
for thresh in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]:
    pred = (illicit_prob > thresh).astype(int)
    r = classification_report(true_labels, pred, labels=[0,1],
                              target_names=['licit','illicit'],
                              output_dict=True, zero_division=0)
    f1 = r['illicit']['f1-score']
    print(f"Threshold {thresh:.2f} → F1: {f1:.4f} | P: {r['illicit']['precision']:.2f} | R: {r['illicit']['recall']:.2f}")
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f"\nBest threshold: {best_thresh} → Final illicit F1: {best_f1:.4f}")
print(f"Weber et al. baseline: 0.6280")

Epoch 10/150, Loss: 0.1977, LR: 0.001000
Epoch 20/150, Loss: 0.1443, LR: 0.001000
Epoch 30/150, Loss: 0.1173, LR: 0.001000
Epoch 40/150, Loss: 0.1063, LR: 0.001000
Epoch 50/150, Loss: 0.0886, LR: 0.001000
Epoch 60/150, Loss: 0.0827, LR: 0.001000
Epoch 70/150, Loss: 0.0775, LR: 0.001000
Epoch 80/150, Loss: 0.0713, LR: 0.001000
Epoch 90/150, Loss: 0.0699, LR: 0.001000
Epoch 100/150, Loss: 0.0661, LR: 0.001000
Epoch 110/150, Loss: 0.0628, LR: 0.001000
Epoch 120/150, Loss: 0.0611, LR: 0.001000
Epoch 130/150, Loss: 0.0601, LR: 0.001000
Epoch 140/150, Loss: 0.0588, LR: 0.001000
Epoch 150/150, Loss: 0.0589, LR: 0.001000
Threshold 0.30 → F1: 0.1431 | P: 0.08 | R: 0.93
Threshold 0.35 → F1: 0.1589 | P: 0.09 | R: 0.89
Threshold 0.40 → F1: 0.1738 | P: 0.10 | R: 0.81
Threshold 0.45 → F1: 0.1985 | P: 0.12 | R: 0.69
Threshold 0.50 → F1: 0.2417 | P: 0.15 | R: 0.56
Threshold 0.55 → F1: 0.2787 | P: 0.21 | R: 0.41
Threshold 0.60 → F1: 0.3072 | P: 0.34 | R: 0.28
Threshold 0.65 → F1: 0.1821 | P: 0.42 | R: 

In [9]:
# save results

import json

gcn_results = {
    "best_hyperparams": {
        "learning_rate": best_lr,
        "weight_decay": best_wd,
        "dropout": best_dropout,
        "hidden_channels": best_hidden,
        "gamma": best_gamma,
        "beta": best_beta
    },
    "final_results": {
        "illicit_f1": class_report_dict['illicit']['f1-score'],
        "illicit_precision": class_report_dict['illicit']['precision'],
        "illicit_recall": class_report_dict['illicit']['recall'],
        "licit_f1": class_report_dict['licit']['f1-score'],
        "accuracy": class_report_dict['accuracy'],
        "validation_loss": total_eval_loss
    },
    "weber_et_al_baseline": 0.628,
    "epochs": 300
}

with open("../reports/gcn_sweep_results.json", "w") as f:
    json.dump(gcn_results, f, indent=2)

print("Saved to reports/gcn_sweep_results.json")

NameError: name 'best_lr' is not defined

In [21]:
# Save best GCN weights for generalization test
torch.save(model.state_dict(), "../reports/gcn_best_weights.pt")
print("Saved GCN weights to reports/gcn_best_weights.pt")

Saved GCN weights to reports/gcn_best_weights.pt
